In [1]:
import pandas as pd

In [4]:
messages = pd.read_csv('SMSSpamCollection.txt', sep='\t',
                           names=["label", "message"])

sep='\t', t means tab, this was the seperator used in our text file, hence why we used '\t'

In [6]:
messages

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


Message is our input feature, and label is our "Dependent" feature



We will do some text pre-processing here, we will convert these words into "Vectors"


In [7]:
messages.shape

(5572, 2)

The overall number of text is 5572

In [12]:
messages["message"].loc[312]

'Think ur smart ? Win £200 this week in our weekly quiz, text PLAY to 85222 now!T&Cs WinnersClub PO BOX 84, M26 3UZ. 16+. GBP1.50/week'

Hahahahah, crazy

In [13]:
messages["message"].loc[5570]

"The guy did some bitching but I acted like i'd be interested in buying something else next week and he gave it to us for free"

lolll

In [15]:
#Libraries for Data cleaning and preprocessing
import re
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/macbook/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [17]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

In [22]:
ps = PorterStemmer() ## Used  for stemming

In [23]:
corpus = []
for i in range(0, len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['message'][i]) ## This line is used to remove all the special characters except capital a-z and small a to z
    
    review = review.lower() #lowering so duplicated words not present

    review = review.split() #Splitting, and for each word we traverse it, use stopwords, and then stemming

    
    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

# 1. Creating a Bag of Words model

In [64]:
from sklearn.feature_extraction.text import CountVectorizer   ## We use countvectorizer to create a Bag of Words
cv = CountVectorizer(max_features=2500, binary = True ,ngram_range= (1,2) )
X = cv.fit_transform(corpus).toarray()

Binary= If a word is present 2 times in a sentence, or in others words greater than 1, the value will be displayed as one only

Max_features = 2500 actuallys gives us the top 2500 most occuring features

ngram_range = (2,2), i'd recommend you to see the documentation for this hyperparameter

In [65]:
X ## Bag of words created, so this is our input feature

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [66]:
X.shape

(5572, 2500)

In [67]:
y=pd.get_dummies(messages['label'])
y=y.iloc[:,1].values

Just doing label encoding, since if you see above, its just spam and ham

In [68]:
y

array([False, False,  True, ..., False, False, False])

In [69]:
# Train Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 0)

80/20 split

In [70]:
X_train

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [71]:
from sklearn.naive_bayes import MultinomialNB
spam_detect_model = MultinomialNB().fit(X_train, y_train)

We use Multinomial Naive Bayes for text classification because it works well with Bag of Words and n-gram features. 


The text messages are converted into numerical features using CountVectorizer, where each feature represents the frequency of a word or phrase.


Multinomial Naive Bayes then learns the probability of each word occurring in spam and non-spam messages and uses these probabilities to classify new messages

In [72]:
#prediction
y_pred=spam_detect_model.predict(X_test)

In [73]:
from sklearn.metrics import accuracy_score,classification_report
from sklearn.metrics import classification_report

In [77]:
score=accuracy_score(y_test,y_pred)
print("score:", score)

score: 0.9865470852017937


In [75]:
print(classification_report(y_pred,y_test))

              precision    recall  f1-score   support

       False       1.00      0.99      0.99       964
        True       0.93      0.98      0.95       151

    accuracy                           0.99      1115
   macro avg       0.96      0.98      0.97      1115
weighted avg       0.99      0.99      0.99      1115



# 2. Creating the Tf-Idf model

In [81]:
from sklearn.feature_extraction.text import TfidfVectorizer
tv = TfidfVectorizer(max_features=2500,ngram_range = (1,2))
X = tv.fit_transform(corpus).toarray()

TF-IDF improves upon Bag of words by reweighting word frequencies based on their importance across documents. 

However, both representations remain sparse and do not capture semantic meaning.

To capture semantic relationships between words, dense word embeddings such as Word2Vec are used

In [83]:
# Train Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 0)

In [89]:
spam_detect_model = MultinomialNB().fit(X_train, y_train)

In [90]:
#prediction
y_pred=spam_detect_model.predict(X_test)

In [91]:
score=accuracy_score(y_test,y_pred)
print("score", score)

score 0.97847533632287


In [94]:
print(classification_report(y_pred,y_test))

              precision    recall  f1-score   support

       False       1.00      0.98      0.99       979
        True       0.85      1.00      0.92       136

    accuracy                           0.98      1115
   macro avg       0.93      0.99      0.95      1115
weighted avg       0.98      0.98      0.98      1115



#### I'd recommend tho to first apply the train test split, and then apply Tf-idf or BoW, as there can be concerns for data leakage

### Using Random forest, MultinomialNB, Logistic Regression  for Tf-idf (OPTIONAL, JUST AN EXPERIMENT)

In [97]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier()
classifier.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [99]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=0,
    n_jobs=-1)

In [101]:
rf.fit(X_train, y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [102]:
y_pred = rf.predict(X_test)

In [103]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9838565022421525
              precision    recall  f1-score   support

       False       0.98      1.00      0.99       955
        True       1.00      0.89      0.94       160

    accuracy                           0.98      1115
   macro avg       0.99      0.94      0.97      1115
weighted avg       0.98      0.98      0.98      1115



The Random Forest model uses multiple decision trees to improve classification performance. 

The number of trees is controlled using n_estimators, while max_depth limits tree complexity to reduce overfitting. 

The max_features parameter introduces randomness by limiting the number of features considered at each split. 

Since the dataset is imbalanced, class_weight="balanced" is used to give equal importance to both classes.

In [107]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

models = {
    "MultinomialNB": MultinomialNB(),
    "LogReg": LogisticRegression(max_iter=2000),
    "RandomForest": RandomForestClassifier(n_estimators=600, random_state=0, n_jobs=-1)
}

for name, model in models.items():
    model.fit(X_train, y_train.ravel())
    pred = model.predict(X_test)
    print(name, "acc =", accuracy_score(y_test.ravel(), pred))

MultinomialNB acc = 0.97847533632287
LogReg acc = 0.9695067264573991
RandomForest acc = 0.9847533632286996


# 3. Word2vec

In [106]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 21.5 MB/s  0:00:012.1 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gensim]━━━━ 1/2 [gensim]


In [108]:
from gensim.models import Word2Vec, KeyedVectors
import gensim.downloader as api
import gensim

### Let's import the google pretrained model first, and then we will create our own. This model is trained to create 300 dimensions

In [109]:
wv = api.load('word2vec-google-news-300') 

[--------------------------------------------------] 1.4% 23.4/1662.8MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[==------------------------------------------------] 4.3% 71.0/1662.8MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[==------------------------------------------------] 5.7% 95.3/1662.8MB downloaded

...

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[==============================================----] 93.3% 1551.4/1662.8MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[================================================--] 97.7% 1624.2/1662.8MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [114]:
wv['Basketball']

array([ 3.93066406e-02, -3.44238281e-02,  2.10937500e-01, -3.20312500e-01,
        1.24023438e-01, -9.91210938e-02,  2.59765625e-01,  3.90625000e-02,
        1.28906250e-01,  9.27734375e-02, -3.33984375e-01,  8.64257812e-02,
        1.38671875e-01,  1.98364258e-03,  3.14453125e-01,  7.86132812e-02,
        2.51953125e-01,  1.91406250e-01,  1.87500000e-01,  8.05664062e-02,
       -3.36914062e-02,  2.15820312e-01,  1.11816406e-01, -3.82812500e-01,
       -7.51953125e-02, -3.27148438e-02,  2.39257812e-01,  1.39648438e-01,
        3.37890625e-01, -3.41796875e-03, -1.42578125e-01, -1.20605469e-01,
        2.79541016e-02, -3.68652344e-02,  7.95898438e-02, -2.63671875e-01,
        2.59765625e-01, -1.37695312e-01, -3.32031250e-01,  2.27539062e-01,
       -1.03515625e-01, -4.29687500e-02,  3.36914062e-02, -4.56542969e-02,
        5.29785156e-02, -2.50000000e-01,  1.78710938e-01, -3.70788574e-03,
       -5.59082031e-02,  3.84765625e-01,  1.00097656e-01,  2.65625000e-01,
        3.81469727e-03,  

#### Pretrained model is not bad, impressive actually

## Applying lemmatization

In [112]:
from nltk.stem import WordNetLemmatizer
lemmatizer=WordNetLemmatizer()

In [115]:
corpus = []
for i in range(0, len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['message'][i])
    review = review.lower()
    review = review.split()
    
    review = [lemmatizer.lemmatize(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

In [116]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

#### Simpleprocess converts a document into a list of lowercased tokens, so we dont have to go throught hte hassle of lowering it , manually

In [118]:
corpus[100]

'please text anymore nothing else say'

In [120]:
words=[]
for sent in corpus:
    sent_token=sent_tokenize(sent)
    for sent in sent_token:
        words.append(simple_preprocess(sent))

words ----> Run this in a cell to see the magic for yourself, if i run it, its gonna take a while to scroll down to current cell hhhh

### These are the list of our unique words

In [122]:
model=gensim.models.Word2Vec(words,window=5,min_count=2)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


### Word2Vec Model Parameters


model = gensim.models.Word2Vec(words, window=5, min_count=2)


words: The input corpus, where each sentence is represented as a list of words. The model learns word relationships based on how words co-occur within sentences.

window = 5: Defines the size of the context window. The model considers up to 5 words before and after a target word, allowing it to capture contextual relationships.

min_count = 2: Ignores words that appear fewer than 2 times in the corpus. This helps remove rare words and noise, improving training efficiency




Also by default, the dimension size is 100 since the vector size is 100, Though i recommened you to read the documentation for this as the hyperparameters are explained in much depth

#### model.wv.index_to_key ---> run this code, to see the unique words, again i am not doing it as i will have to scroll down a lot to reach to the current cell lol

In [135]:
model.corpus_count ## Our total vocabulary size

5564

In [136]:
model.epochs

5

In [132]:
model.wv.similar_by_word('early')

[('go', 0.9992079138755798),
 ('day', 0.9992030262947083),
 ('one', 0.999199390411377),
 ('come', 0.9991967678070068),
 ('got', 0.9991730451583862),
 ('like', 0.9991670250892639),
 ('dont', 0.9991625547409058),
 ('meet', 0.99915611743927),
 ('need', 0.999152421951294),
 ('thing', 0.9991427063941956)]

In [133]:
#Haha google pretrained model was way better lol

## 4. Avg Word2Vec

In [149]:
import numpy as np

In [150]:
def avg_word2vec(doc):
    # remove out-of-vocabulary words
    #sent = [word for word in doc if word in model.wv.index_to_key]
    #print(sent)
    
    return np.mean([model.wv[word] for word in doc if word in model.wv.index_to_key],axis=0)
                #or [np.zeros(len(model.wv.index_to_key))], axis=0)

This function converts a document into a single vector by averaging the Word2Vec embeddings of the words it contains.

doc
Represents a document as a list of words.

The function first ignores out-of-vocabulary words, meaning words that are not present in the Word2Vec model’s learned vocabulary.

For each valid word in the document, the corresponding Word2Vec vector is retrieved from the model.

All retrieved word vectors are then averaged using np.mean along axis 0, producing a single fixed-length vector that represents the entire document.

The commented fallback line (using zeros) is intended to handle cases where none of the words in the document exist in the model’s vocabulary, preventing errors.

In [151]:
!pip install tqdm

In [152]:
from tqdm import tqdm

In [ ]:
#apply for the entire sentences
X=[]
for i in tqdm(range(len(words))):
    print("Hello",i)
    X.append(avg_word2vec(words[i]))


In [154]:
## Run this code and you will see the whole list lol


In [155]:
type(X)

list